# Skin Lesion Classification (HAM10000) – Full Pipeline
A) Dataset → B) K-means segmentation → C/D) Features → E) Feature impact → F) Graph → G) Models → H) Result analysis

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Put the HAM10000 data in Google Drive (folders or ZIPs of `HAM10000_images_part_1`, `HAM10000_images_part_2`, plus `HAM10000_metadata.csv`).

## 1. Setup

In [ ]:
!pip -q install opencv-python-headless networkx
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Get the data
Edit `DRIVE_DIR` to the Drive folder containing the dataset. ZIPs are extracted automatically to fast local storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/HAM10000'   # <-- change this

import os, glob, zipfile
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('extracting', z)
    zipfile.ZipFile(z).extractall(DATA)
# folders (not zipped): copy locally for speed
for d in glob.glob(f'{DRIVE_DIR}/HAM10000_images_part_*'):
    if os.path.isdir(d):
        !cp -r "{d}" {DATA}/
META = (glob.glob(f'{DRIVE_DIR}/**/HAM10000_metadata*.csv', recursive=True) + glob.glob(f'{DATA}/**/HAM10000_metadata*.csv', recursive=True))[0]
IMG_DIRS = [DATA]   # script searches recursively for all .jpg files
print('metadata:', META)
print('images found:', sum(len(f) for _,_,f in os.walk(DATA)))

## 3. Pipeline code
This cell writes the full pipeline (steps A–H) to `skin_lesion_pipeline.py`.

In [ ]:
%%writefile skin_lesion_pipeline.py
#!/usr/bin/env python3
"""
Skin-lesion classification pipeline (HAM10000) following the flow diagram:

 (A) Dataset demonstration        -> load metadata, binary benign/malignant labels
 (B) K-means (K=2) segmentation   -> mask + segmented lesion
 (C) Feature extraction           -> contours, SIFT, ellipse, Harris, asymmetry ...
 (D) Computerized features        -> 9 hand-crafted features
 (E) Feature impact analysis      -> ANOVA F-test, SelectKBest
 (F) Graph generation             -> correlation graph between feature nodes
 (G) Model analysis               -> Graph (GNN, GAT) / Feature (CNN, ML, Ensemble)
                                     / Image (AlexNet, EfficientNet-B3, VGG16, InceptionV3)
 (H) Result analysis              -> metrics, loss & AUC curves, statistics,
                                     linear regression, 5-fold CV, ablation,
                                     graph-threshold experiment

Usage
-----
python skin_lesion_pipeline.py \
    --meta HAM10000_metadata.csv \
    --img_dirs HAM10000_images_part_1 HAM10000_images_part_2 \
    --out results

Quick test on a subset, no image CNNs:
python skin_lesion_pipeline.py --meta ... --img_dirs ... --limit 1500 --skip_image_models

Requirements
------------
pip install numpy pandas opencv-contrib-python scikit-learn scipy matplotlib \
            networkx torch torchvision joblib
(opencv >= 4.4 has SIFT in the main package; contrib is fine as well.)
"""
import argparse
import copy
import json
import os
import warnings

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from joblib import Parallel, delayed
from scipy import stats
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              VotingClassifier)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, Dataset, TensorDataset

warnings.filterwarnings("ignore")

FEATURES = ["sift_keypoints", "wh_ratio", "ecl_ratio", "harris_corners",
            "circularity", "solidity", "asymmetry_score", "compactness_index",
            "color_variation"]
IMG_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# HAM10000 diagnosis -> binary label (1 = malignant)
MALIGNANT = {"mel", "bcc", "akiec"}


# =============================================================================
# (A) Dataset
# =============================================================================
def load_dataset(meta_csv, img_dirs, label_col="dx", malignant=MALIGNANT,
                 limit=None, seed=42):
    meta = pd.read_csv(meta_csv)
    # index every image file in all given folders
    index = {}
    for d in img_dirs:
        for root, _, files in os.walk(d):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    index[os.path.splitext(f)[0]] = os.path.join(root, f)
    meta["path"] = meta["image_id"].map(index)
    meta = meta.dropna(subset=["path"]).reset_index(drop=True)
    meta["label"] = meta[label_col].isin(malignant).astype(int)
    if limit and limit < len(meta):  # stratified subset for quick experiments
        meta, _ = train_test_split(meta, train_size=limit, stratify=meta["label"],
                                   random_state=seed)
        meta = meta.reset_index(drop=True)
    print(f"[A] {len(meta)} images | benign={int((meta.label==0).sum())} "
          f"malignant={int((meta.label==1).sum())}")
    return meta


# =============================================================================
# (B) Segmentation with K-means (K = 2)
# =============================================================================
def segment_kmeans(img_bgr, k=2):
    img = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    cv2.setRNGSeed(0)
    _, labels, centers = cv2.kmeans(lab, k, None, crit, 3, cv2.KMEANS_PP_CENTERS)
    lesion_cluster = int(np.argmin(centers[:, 0]))  # lesion = darker cluster
    mask = (labels.reshape(IMG_SIZE, IMG_SIZE) == lesion_cluster).astype(np.uint8) * 255

    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, ker)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, ker)
    n, cc, st, _ = cv2.connectedComponentsWithStats(mask)
    if n > 1:  # keep largest component
        idx = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
        mask = ((cc == idx) * 255).astype(np.uint8)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, cnts, -1, 255, -1)  # fill holes
    segmented = cv2.bitwise_and(img, img, mask=filled)
    return img, filled, segmented


# =============================================================================
# (C)+(D) Feature extraction
# =============================================================================
def asymmetry_score(mask, contour):
    """0 = perfectly symmetric, 1 = totally asymmetric. Align principal axis,
    flip horizontally / vertically, average (1 - IoU)."""
    M = cv2.moments(contour)
    cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
    ys, xs = np.nonzero(mask)
    cov = np.cov(np.vstack([xs - cx, ys - cy]))
    _, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    h, w = mask.shape
    R = cv2.getRotationMatrix2D((cx, cy), ang, 1.0)
    R[0, 2] += w / 2 - cx
    R[1, 2] += h / 2 - cy
    rot = cv2.warpAffine(mask, R, (w, h), flags=cv2.INTER_NEAREST) > 0
    scores = []
    for flip in (rot[:, ::-1], rot[::-1, :]):
        inter = np.logical_and(rot, flip).sum()
        union = np.logical_or(rot, flip).sum()
        scores.append(1 - inter / max(union, 1))
    return float(np.mean(scores))


def extract_features(path, return_vis=False):
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None
    img, mask, seg = segment_kmeans(img_bgr)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    area, perim = cv2.contourArea(cnt), cv2.arcLength(cnt, True)
    if area < 100 or len(cnt) < 5:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # a. SIFT key points inside the lesion
    sift = cv2.SIFT_create()
    kps = sift.detect(gray, mask)
    # b. width/height ratio of bounding box
    x, y, w, h = cv2.boundingRect(cnt)
    wh_ratio = w / max(h, 1)
    # c. ECL ratio: major/minor axis of the fitted ellipse
    (_, (d1, d2), _) = cv2.fitEllipse(cnt)
    ecl_ratio = max(d1, d2) / max(min(d1, d2), 1e-6)
    # d. Harris corners inside the lesion
    harris = cv2.cornerHarris(np.float32(gray), 2, 3, 0.04)
    harris_n = int(((harris > 0.01 * harris.max()) & (mask > 0)).sum())
    # e. circularity, f. solidity
    circularity = 4 * np.pi * area / (perim ** 2 + 1e-6)
    hull_area = cv2.contourArea(cv2.convexHull(cnt))
    solidity = area / max(hull_area, 1e-6)
    # g. asymmetry, h. compactness index
    asym = asymmetry_score(mask, cnt)
    compactness = perim ** 2 / area
    # colour variation: mean per-channel std inside the lesion
    color_var = float(np.mean(img[mask > 0].std(axis=0)))

    feats = dict(sift_keypoints=len(kps), wh_ratio=wh_ratio, ecl_ratio=ecl_ratio,
                 harris_corners=harris_n, circularity=circularity,
                 solidity=solidity, asymmetry_score=asym,
                 compactness_index=compactness, color_variation=color_var)
    if return_vis:
        vis = img.copy()
        cv2.drawContours(vis, [cnt], -1, (0, 255, 0), 2)
        cv2.drawContours(vis, [cv2.convexHull(cnt)], -1, (0, 0, 255), 1)
        for k in kps:
            cv2.circle(vis, (int(k.pt[0]), int(k.pt[1])), 2, (0, 165, 255), -1)
        return feats, (img, mask, seg, vis)
    return feats


def extract_all(meta, cache, n_jobs=-1):
    if os.path.exists(cache):
        print(f"[C/D] loading cached features from {cache}")
        return pd.read_csv(cache)
    print("[C/D] extracting features ...")
    rows = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(extract_features)(p) for p in meta["path"])
    keep = [i for i, r in enumerate(rows) if r is not None]
    df = pd.DataFrame([rows[i] for i in keep])
    df.insert(0, "image_id", meta.loc[keep, "image_id"].values)
    df["path"] = meta.loc[keep, "path"].values
    df["label"] = meta.loc[keep, "label"].values
    df.to_csv(cache, index=False)
    return df


def save_visual_panels(df, out, n=2):
    """Figures (A)(B)(C): benign/malignant samples, mask, segmentation, contours."""
    fig, ax = plt.subplots(2 * n, 4, figsize=(10, 5 * n))
    r = 0
    for lab in (0, 1):
        for p in df[df.label == lab]["path"].head(n):
            f, (img, mask, seg, vis) = extract_features(p, return_vis=True)
            for a, im, t in zip(ax[r], [img, mask, seg, vis],
                                ["Original", "Generated mask", "Segmented", "Features"]):
                a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im.ndim == 3 else im,
                         cmap="gray")
                a.set_title(f"{t} ({'malignant' if lab else 'benign'})", fontsize=8)
                a.axis("off")
            r += 1
    plt.tight_layout()
    plt.savefig(os.path.join(out, "B_C_segmentation_features.png"), dpi=130)
    plt.close()


# =============================================================================
# (E) Feature impact analysis
# =============================================================================
def feature_impact(df, out, k=7):
    X, y = df[FEATURES].values, df["label"].values
    sel = SelectKBest(f_classif, k=min(k, len(FEATURES))).fit(X, y)
    table = pd.DataFrame({"feature": FEATURES, "F_score": sel.scores_,
                          "p_value": sel.pvalues_, "selected": sel.get_support()})
    table = table.sort_values("F_score", ascending=False)
    table.to_csv(os.path.join(out, "E_anova_selectkbest.csv"), index=False)
    plt.figure(figsize=(7, 4))
    plt.barh(table.feature[::-1], table.F_score[::-1],
             color=["tab:green" if s else "tab:gray" for s in table.selected[::-1]])
    plt.xlabel("ANOVA F-score"); plt.title(f"Feature impact (SelectKBest k={k})")
    plt.tight_layout(); plt.savefig(os.path.join(out, "E_feature_impact.png"), dpi=130)
    plt.close()
    print("[E]\n", table.to_string(index=False))
    return list(table.feature[table.selected])


# =============================================================================
# (F) Graph generation
# =============================================================================
def build_adjacency(X, thr):
    """Nodes = features, edge weight = |Pearson corr| if >= thr."""
    C = np.nan_to_num(np.corrcoef(X.T))
    A = np.abs(C) * (np.abs(C) >= thr)
    np.fill_diagonal(A, 0)
    return A


def draw_graph(df, cols, thr, out):
    """Figure (F): feature graph incl. Class node (correlation edges)."""
    Xc = df[cols + ["label"]].corr()
    G = nx.Graph()
    names = cols + ["label"]
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if abs(Xc.loc[a, b]) >= thr:
                G.add_edge(a, b, weight=round(Xc.loc[a, b], 2))
    G.add_nodes_from(names)
    pos = nx.spring_layout(G, seed=1)
    plt.figure(figsize=(8, 6))
    nx.draw_networkx(G, pos, node_color="tab:blue", font_size=7, node_size=500,
                     font_color="black")
    nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "weight"), font_size=6)
    plt.title(f"Feature graph (|corr| >= {thr})"); plt.axis("off")
    plt.savefig(os.path.join(out, "F_graph.png"), dpi=130, bbox_inches="tight")
    plt.close()


def norm_adj(A):
    A = A + np.eye(len(A))
    d = A.sum(1) ** -0.5
    return (d[:, None] * A * d[None, :]).astype(np.float32)


# =============================================================================
# (G) Models
# =============================================================================
class DenseGCN(nn.Module):
    """Each sample = graph with one node per feature (node input = scalar value).
    Shared adjacency from feature correlations (dense GCN, no PyG needed)."""
    def __init__(self, A, hid=32, layers=2, drop=0.3):
        super().__init__()
        n = len(A)
        self.register_buffer("A", torch.tensor(norm_adj(A)))
        self.inp = nn.Linear(1, hid)
        self.emb = nn.Parameter(torch.randn(n, hid) * 0.1)  # node identity
        self.convs = nn.ModuleList([nn.Linear(hid, hid) for _ in range(layers)])
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Dropout(drop),
                                  nn.Linear(hid, 2))

    def forward(self, x):                      # x: B x N
        h = self.inp(x.unsqueeze(-1)) + self.emb
        for conv in self.convs:
            h = self.drop(F.relu(self.A @ conv(h)))
        return self.head(h.mean(1))


class GATLayer(nn.Module):
    def __init__(self, i, o, heads=2, drop=0.2):
        super().__init__()
        self.H, self.O = heads, o
        self.W = nn.Linear(i, o * heads, bias=False)
        self.a_src = nn.Parameter(torch.randn(heads, o) * 0.1)
        self.a_dst = nn.Parameter(torch.randn(heads, o) * 0.1)
        self.drop = nn.Dropout(drop)

    def forward(self, h, mask):
        B, N, _ = h.shape
        z = self.W(h).view(B, N, self.H, self.O)
        es, ed = (z * self.a_src).sum(-1), (z * self.a_dst).sum(-1)    # B,N,H
        e = F.leaky_relu(es.unsqueeze(2) + ed.unsqueeze(1), 0.2)       # B,N,N,H
        e = e.masked_fill(~mask.view(1, N, N, 1), -1e9)
        a = self.drop(torch.softmax(e, dim=2))
        out = torch.einsum("bijh,bjho->biho", a, z).reshape(B, N, self.H * self.O)
        return F.elu(out)


class DenseGAT(nn.Module):
    def __init__(self, A, hid=32, heads=2, layers=2, drop=0.3):
        super().__init__()
        n = len(A)
        self.register_buffer("mask", torch.tensor((A > 0) | np.eye(n, dtype=bool)))
        self.inp = nn.Linear(1, hid)
        self.emb = nn.Parameter(torch.randn(n, hid) * 0.1)
        self.gats = nn.ModuleList([GATLayer(hid, hid // heads, heads) for _ in range(layers)])
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Dropout(drop),
                                  nn.Linear(hid, 2))

    def forward(self, x):
        h = self.inp(x.unsqueeze(-1)) + self.emb
        for g in self.gats:
            h = self.drop(g(h, self.mask))
        return self.head(h.mean(1))


class FeatureCNN(nn.Module):
    """1-D CNN over the feature vector."""
    def __init__(self, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(),
            nn.Dropout(drop), nn.Linear(64, 2))

    def forward(self, x):
        return self.net(x.unsqueeze(1))


# ---- image based models -----------------------------------------------------
class ImgDS(Dataset):
    MEAN = np.array([0.485, 0.456, 0.406], np.float32)
    STD = np.array([0.229, 0.224, 0.225], np.float32)

    def __init__(self, paths, labels, size, train):
        self.p, self.y, self.s, self.train = list(paths), list(labels), size, train

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.p[i]), cv2.COLOR_BGR2RGB)
        im = cv2.resize(im, (self.s, self.s))
        if self.train:
            if np.random.rand() < .5: im = im[:, ::-1]
            if np.random.rand() < .5: im = im[::-1]
        im = (im.astype(np.float32) / 255 - self.MEAN) / self.STD
        return torch.tensor(im.transpose(2, 0, 1).copy()), self.y[i]


def build_image_model(name, finetune_all=False):
    import torchvision.models as tv
    if name == "AlexNet":
        m = tv.alexnet(weights="DEFAULT"); size = 224
    elif name == "EfficientNet-B3":
        m = tv.efficientnet_b3(weights="DEFAULT"); size = 300
    elif name == "VGG16":
        m = tv.vgg16(weights="DEFAULT"); size = 224
    elif name == "InceptionV3":
        m = tv.inception_v3(weights="DEFAULT"); size = 299
        m.aux_logits, m.AuxLogits = False, None
    else:
        raise ValueError(name)
    if not finetune_all:
        for p in m.parameters():
            p.requires_grad = False
    if name in ("AlexNet", "VGG16"):
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, 2)
    elif name == "EfficientNet-B3":
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    else:
        m.fc = nn.Linear(m.fc.in_features, 2)
    return m, size


# ---- training / evaluation ---------------------------------------------------
def train_torch(model, train_dl, val_dl, epochs=60, lr=1e-3, class_w=None, patience=12):
    model.to(DEVICE)
    w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE) if class_w is not None else None
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                           lr=lr, weight_decay=1e-4)
    hist = {"train_loss": [], "val_loss": [], "val_auc": []}
    best, best_auc, bad = None, -1, 0
    for ep in range(epochs):
        model.train(); tl, n = 0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            tl += loss.item() * len(yb); n += len(yb)
        probs, ys, vl = predict(model, val_dl, crit)
        auc = roc_auc_score(ys, probs) if len(set(ys)) > 1 else 0.5
        hist["train_loss"].append(tl / n); hist["val_loss"].append(vl); hist["val_auc"].append(auc)
        if auc > best_auc:
            best_auc, best, bad = auc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best)
    return hist


@torch.no_grad()
def predict(model, dl, crit=None):
    model.eval(); P, Y, tl, n = [], [], 0, 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        if crit is not None:
            tl += crit(out, yb).item() * len(yb); n += len(yb)
        P.append(torch.softmax(out, 1)[:, 1].cpu().numpy()); Y.append(yb.cpu().numpy())
    return np.concatenate(P), np.concatenate(Y), (tl / max(n, 1))


def metrics(y, p):
    pred = (p >= 0.5).astype(int)
    tn = ((pred == 0) & (y == 0)).sum(); fp = ((pred == 1) & (y == 0)).sum()
    return dict(accuracy=accuracy_score(y, pred),
                precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0),
                specificity=tn / max(tn + fp, 1),
                f1=f1_score(y, pred, zero_division=0),
                auc=roc_auc_score(y, p) if len(set(y)) > 1 else float("nan"))


def class_weights(y):
    c = np.bincount(y, minlength=2)
    return (len(y) / (2 * np.maximum(c, 1))).tolist()


def tab_loader(X, y, bs=128, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)


def fit_eval_neural(kind, tr, va, te, cols, thr=0.2, epochs=80):
    """kind in {GNN, GAT, CNN}. tr/va/te are DataFrames with `label`."""
    sc = StandardScaler().fit(tr[cols])
    Xtr, Xva, Xte = [sc.transform(d[cols]) for d in (tr, va, te)]
    ytr, yva, yte = tr.label.values, va.label.values, te.label.values
    if kind == "GNN":
        model = DenseGCN(build_adjacency(Xtr, thr))
    elif kind == "GAT":
        model = DenseGAT(build_adjacency(Xtr, thr))
    else:
        model = FeatureCNN()
    hist = train_torch(model, tab_loader(Xtr, ytr, shuffle=True), tab_loader(Xva, yva),
                       epochs=epochs, class_w=class_weights(ytr))
    p, y, _ = predict(model, tab_loader(Xte, yte))
    return metrics(y, p), hist, (y, p)


def fit_eval_ml(name, tr, va, te, cols):
    tr = pd.concat([tr, va])  # sklearn models need no validation split
    sc = StandardScaler().fit(tr[cols])
    Xtr, Xte = sc.transform(tr[cols]), sc.transform(te[cols])
    rf = RandomForestClassifier(300, class_weight="balanced", random_state=0, n_jobs=-1)
    svm = SVC(probability=True, class_weight="balanced", random_state=0)
    lr = LogisticRegression(class_weight="balanced", max_iter=1000)
    gb = GradientBoostingClassifier(random_state=0)
    models = {"RandomForest": rf, "SVM": svm, "LogReg": lr, "GradBoost": gb,
              "Ensemble": VotingClassifier([("rf", rf), ("svm", svm), ("lr", lr), ("gb", gb)],
                                           voting="soft")}
    m = models[name].fit(Xtr, tr.label.values)
    p = m.predict_proba(Xte)[:, 1]
    return metrics(te.label.values, p), (te.label.values, p)


def fit_eval_image(name, tr, va, te, epochs, bs, finetune_all):
    model, size = build_image_model(name, finetune_all)
    mk = lambda d, t: DataLoader(ImgDS(d.path, d.label, size, t), batch_size=bs,
                                 shuffle=t, num_workers=2)
    hist = train_torch(model, mk(tr, True), mk(va, False), epochs=epochs, lr=1e-4,
                       class_w=class_weights(tr.label.values), patience=4)
    p, y, _ = predict(model, mk(te, False))
    return metrics(y, p), hist, (y, p)


# =============================================================================
# (H) Result analysis helpers
# =============================================================================
def plot_curves(hists, rocs, out):
    n = len(hists)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in hists.items():
        ax[0].plot(h["val_loss"], label=name)
    ax[0].set_title("Validation loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=6)
    for name, h in hists.items():
        ax[1].plot(h["val_auc"], label=name)
    ax[1].set_title("Validation AUC"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(os.path.join(out, "H_loss_auc_curves.png"), dpi=130)
    plt.close()
    plt.figure(figsize=(5.5, 5))
    for name, (y, p) in rocs.items():
        fpr, tpr, _ = roc_curve(y, p)
        plt.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y, p):.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=.5); plt.legend(fontsize=6)
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC curves")
    plt.savefig(os.path.join(out, "H_roc.png"), dpi=130, bbox_inches="tight"); plt.close()


def linear_regression_analysis(df, out):
    """Univariate linear regression of the class on every feature + multivariate R^2."""
    from sklearn.linear_model import LinearRegression
    rows = []
    for f in FEATURES:
        r = stats.linregress(df[f], df.label)
        rows.append(dict(feature=f, slope=r.slope, r=r.rvalue, r2=r.rvalue ** 2, p_value=r.pvalue))
    t = pd.DataFrame(rows).sort_values("r2", ascending=False)
    Xs = StandardScaler().fit_transform(df[FEATURES])
    t.attrs["multivariate_r2"] = LinearRegression().fit(Xs, df.label).score(Xs, df.label)
    t.to_csv(os.path.join(out, "H_linear_regression.csv"), index=False)
    print(f"[H] linear regression multivariate R2 = {t.attrs['multivariate_r2']:.3f}")
    return t


def cross_validate(df, cols, out, folds=5, epochs=60):
    skf = StratifiedKFold(folds, shuffle=True, random_state=42)
    rows = []
    for k, (a, b) in enumerate(skf.split(df, df.label)):
        trn, te = df.iloc[a], df.iloc[b]
        tr, va = train_test_split(trn, test_size=0.1, stratify=trn.label, random_state=k)
        for kind in ("GNN", "GAT", "CNN"):
            m, _, _ = fit_eval_neural(kind, tr, va, te, cols, epochs=epochs)
            rows.append(dict(fold=k + 1, model=kind, **m))
        m, _ = fit_eval_ml("Ensemble", tr, va, te, cols)
        rows.append(dict(fold=k + 1, model="Ensemble", **m))
    r = pd.DataFrame(rows)
    r.to_csv(os.path.join(out, "H_5fold_cv.csv"), index=False)
    summ = r.groupby("model")[["accuracy", "f1", "auc"]].agg(["mean", "std"])
    summ.to_csv(os.path.join(out, "H_5fold_summary.csv"))
    print("[H] 5-fold CV\n", summ.round(4).to_string())


def ablation_study(tr, va, te, cols, out, epochs=60):
    rows = [dict(setting="all features", **fit_eval_neural("GNN", tr, va, te, cols, epochs=epochs)[0])]
    for c in cols:  # drop one feature (node) at a time
        rest = [x for x in cols if x != c]
        rows.append(dict(setting=f"- {c}", **fit_eval_neural("GNN", tr, va, te, rest, epochs=epochs)[0]))
    rows.append(dict(setting="no edges (thr=2)",
                     **fit_eval_neural("GNN", tr, va, te, cols, thr=2.0, epochs=epochs)[0]))
    t = pd.DataFrame(rows); t.to_csv(os.path.join(out, "H_ablation.csv"), index=False)
    print("[H] ablation\n", t[["setting", "accuracy", "f1", "auc"]].round(4).to_string(index=False))


def threshold_experiment(tr, va, te, cols, out, epochs=60,
                         thrs=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7)):
    rows = []
    for t in thrs:
        for kind in ("GNN", "GAT"):
            m, _, _ = fit_eval_neural(kind, tr, va, te, cols, thr=t, epochs=epochs)
            rows.append(dict(threshold=t, model=kind, **m))
    r = pd.DataFrame(rows); r.to_csv(os.path.join(out, "H_threshold_experiment.csv"), index=False)
    plt.figure(figsize=(5, 4))
    for kind, g in r.groupby("model"):
        plt.plot(g.threshold, g.auc, marker="o", label=kind)
    plt.xlabel("correlation threshold"); plt.ylabel("test AUC"); plt.legend()
    plt.title("Graph threshold experiment"); plt.tight_layout()
    plt.savefig(os.path.join(out, "H_threshold_experiment.png"), dpi=130); plt.close()
    print("[H] threshold experiment\n", r[["threshold", "model", "accuracy", "auc"]].round(4).to_string(index=False))


# =============================================================================
# main
# =============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta", required=True, help="HAM10000_metadata.csv (or similar)")
    ap.add_argument("--img_dirs", nargs="+", required=True)
    ap.add_argument("--out", default="results")
    ap.add_argument("--label_col", default="dx")
    ap.add_argument("--malignant", nargs="+", default=sorted(MALIGNANT),
                    help="labels in label_col counted as malignant (use for other datasets)")
    ap.add_argument("--limit", type=int, default=None, help="use a stratified subset")
    ap.add_argument("--k_best", type=int, default=7)
    ap.add_argument("--graph_thr", type=float, default=0.2)
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--img_epochs", type=int, default=8)
    ap.add_argument("--img_bs", type=int, default=32)
    ap.add_argument("--finetune_all", action="store_true")
    ap.add_argument("--skip_image_models", action="store_true")
    ap.add_argument("--skip_extra", action="store_true", help="skip CV / ablation / threshold")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    np.random.seed(a.seed); torch.manual_seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    print("device:", DEVICE)

    # (A)-(D)
    meta = load_dataset(a.meta, a.img_dirs, a.label_col, set(a.malignant), a.limit, a.seed)
    df = extract_all(meta, os.path.join(a.out, "features.csv"))
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
    save_visual_panels(df, a.out)

    # (E)
    selected = feature_impact(df, a.out, a.k_best)
    cols = [c for c in FEATURES if c in selected]

    # (F)
    draw_graph(df, cols, a.graph_thr, a.out)

    # split 70/15/15
    trn, te = train_test_split(df, test_size=0.15, stratify=df.label, random_state=a.seed)
    tr, va = train_test_split(trn, test_size=0.15 / 0.85, stratify=trn.label, random_state=a.seed)
    print(f"split: train={len(tr)} val={len(va)} test={len(te)}")

    # (G)
    results, hists, rocs = {}, {}, {}
    for kind in ("GNN", "GAT", "CNN"):
        m, h, r = fit_eval_neural(kind, tr, va, te, cols, a.graph_thr, a.epochs)
        results[f"{kind} ({'graph' if kind != 'CNN' else 'feature'})"] = m
        hists[kind], rocs[kind] = h, r
    for name in ("RandomForest", "SVM", "LogReg", "GradBoost", "Ensemble"):
        m, r = fit_eval_ml(name, tr, va, te, cols)
        results[f"{name} (feature)"] = m; rocs[name] = r
    if not a.skip_image_models:
        for name in ("AlexNet", "EfficientNet-B3", "VGG16", "InceptionV3"):
            print(f"[G] training {name} ...")
            m, h, r = fit_eval_image(name, tr, va, te, a.img_epochs, a.img_bs, a.finetune_all)
            results[f"{name} (image)"] = m; hists[name], rocs[name] = h, r

    # (H)
    res = pd.DataFrame(results).T.sort_values("auc", ascending=False)
    res.to_csv(os.path.join(a.out, "H_model_comparison.csv"))
    print("\n[H] Model comparison (test set)\n", res.round(4).to_string())
    print("Best model:", res.index[0])
    plot_curves(hists, rocs, a.out)
    linear_regression_analysis(df, a.out)

    if not a.skip_extra:
        cross_validate(df, cols, a.out, epochs=min(a.epochs, 60))
        ablation_study(tr, va, te, cols, a.out, epochs=min(a.epochs, 60))
        threshold_experiment(tr, va, te, cols, a.out, epochs=min(a.epochs, 60))

    json.dump(vars(a), open(os.path.join(a.out, "run_config.json"), "w"), indent=2)
    print("\nDone. Outputs in", a.out)


if __name__ == "__main__":
    main()


## 4. Quick test run (subset, no image CNNs)
Checks everything works in a few minutes.

In [ ]:
!python skin_lesion_pipeline.py --meta "{META}" --img_dirs {' '.join(IMG_DIRS)} --out results_quick --limit 1500 --skip_image_models --epochs 40

## 5. Full run (all models + 5-fold CV, ablation, threshold experiment)
Feature extraction is cached in `results/features.csv`, so re-runs are faster. Add `--skip_image_models` to skip AlexNet/EfficientNet/VGG16/InceptionV3.

In [ ]:
!python skin_lesion_pipeline.py --meta "{META}" --img_dirs {' '.join(IMG_DIRS)} --out results --epochs 80 --img_epochs 8

## 6. View results

In [ ]:
import pandas as pd
from IPython.display import display, Image
R = 'results'   # use 'results_quick' for the quick run
for f in ['E_anova_selectkbest','H_model_comparison','H_linear_regression','H_5fold_summary','H_ablation','H_threshold_experiment']:
    p = f'{R}/{f}.csv'
    if os.path.exists(p):
        print('\n==', f); display(pd.read_csv(p).round(4))
for f in ['B_C_segmentation_features','E_feature_impact','F_graph','H_loss_auc_curves','H_roc','H_threshold_experiment']:
    p = f'{R}/{f}.png'
    if os.path.exists(p): display(Image(p))

## 7. Save results back to Drive

In [ ]:
!cp -r results /content/drive/MyDrive/HAM10000_results